# VinDr-Mammo Multi-Objective Optimization Pipeline

**Complete workflow with stratified dataset and preprocessing**

## Workflow:
1. Load stratified dataset (`stratified_selection.csv`)
2. Convert DICOM → PNG using preprocessing pipeline
3. Train/val/test split (patient-level)
4. Run NSGA-III optimization
5. Analyze results

---

In [ ]:
# ============================================================================
# MODIFY THESE PARAMETERS BEFORE RUNNING OPTIMIZATION
# ============================================================================

# Choose your configuration preset:
# - "testing": Fast test run (~30 min)
# - "default": Balanced run (~6 days)
# - "production": Full optimization (~46 days)
# - "custom": Set your own values below

CONFIG_PRESET = "testing"  # Change this!

# Preset configurations
PRESETS = {
    "testing": {
        "pop_size": 6,
        "n_generations": 3,
        "max_epochs": 10,
        "batch_size": 32,
        "early_stopping_patience": 3
    },
    "default": {
        "pop_size": 24,
        "n_generations": 50,
        "max_epochs": 100,
        "batch_size": 64,
        "early_stopping_patience": 15
    },
    "production": {
        "pop_size": 48,
        "n_generations": 100,
        "max_epochs": 200,
        "batch_size": 128,
        "early_stopping_patience": 20
    },
    "custom": {
        # Set your own values here
        "pop_size": 12,
        "n_generations": 10,
        "max_epochs": 50,
        "batch_size": 64,
        "early_stopping_patience": 10
    }
}

# Get selected configuration
config = PRESETS[CONFIG_PRESET]

# Display selected configuration
print("=" * 70)
print(f"SELECTED CONFIGURATION: {CONFIG_PRESET.upper()}")
print("=" * 70)
print(f"\nOptimization Parameters:")
print(f"  Population Size:          {config['pop_size']}")
print(f"  Generations:              {config['n_generations']}")
print(f"  Max Epochs per Model:     {config['max_epochs']}")
print(f"  Batch Size:               {config['batch_size']}")
print(f"  Early Stopping Patience:  {config['early_stopping_patience']}")

# Calculate time estimates
models_per_gen = config['pop_size']
total_models = models_per_gen * config['n_generations']
avg_epochs = config['max_epochs'] * 0.6  # Assume early stopping at 60%
time_per_model = (avg_epochs * 16384 / config['batch_size']) / 60  # minutes
time_per_gen = time_per_model * models_per_gen / 60  # hours
total_time = time_per_gen * config['n_generations']  # hours

print(f"\nEstimated Time:")
print(f"  Per Model:      ~{time_per_model:.1f} minutes")
print(f"  Per Generation: ~{time_per_gen:.1f} hours")
print(f"  Total Time:     ~{total_time:.1f} hours ({total_time/24:.1f} days)")
print(f"  Total Models:   {total_models}")

print("\n" + "=" * 70)
print("These values will be applied when you run optimization.")
print("To change, modify CONFIG_PRESET above and re-run this cell.")
print("=" * 70)

## Configuration: Modify Optimization Parameters

**Customize these values before running optimization:**

### 🎛️ Quick Settings:

| Setting | Default | Testing | Production |
|---------|---------|---------|------------|
| **Population Size** | 24 | 6 | 48 |
| **Generations** | 50 | 3 | 100 |
| **Max Epochs** | 100 | 10 | 200 |
| **Batch Size** | 64 | 32 | 128 |

**Time estimates:**
- **Testing** (6 pop, 3 gen, 10 epochs): ~30 minutes
- **Default** (24 pop, 50 gen, 100 epochs): ~5.8 days
- **Production** (48 pop, 100 gen, 200 epochs): ~46 days

**Run the cell below to set your configuration:**

---

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paths
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
STRATIFIED_CSV = f'{BASE_DIR}/metadata/stratified_selection.csv'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'
OPTIMIZATION_DIR = '/content/drive/MyDrive/vindr_optimization'

print("✅ Google Drive mounted")
print(f"📁 Base directory: {BASE_DIR}")
print(f"📊 Stratified CSV: {STRATIFIED_CSV}")
print(f"🖼️  Preprocessed PNGs: {PREPROCESSED_DIR}")
print(f"📈 Optimization output: {OPTIMIZATION_DIR}")

## Step 3: Install Dependencies

In [ ]:
!pip install -q pydicom scikit-image pymoo pandas matplotlib pillow opencv-python
print("✅ Dependencies installed")

## Step 4: Clone Repository (if needed)

In [ ]:
import os

if not os.path.exists('mammography-multiobjective-optimization'):
    !git clone https://github.com/dtobi59/mammography-multiobjective-optimization.git
    %cd mammography-multiobjective-optimization
    !pip install -q -r requirements.txt
else:
    %cd mammography-multiobjective-optimization
    !git pull

print("✅ Repository ready")

## Step 5: Load Stratified Dataset

In [ ]:
import pandas as pd
from pathlib import Path

# Load stratified selection
print("📊 Loading stratified dataset...")
stratified_df = pd.read_csv(STRATIFIED_CSV)

print(f"\n✅ Loaded {len(stratified_df)} images")
print(f"\n📋 Label Distribution:")
print(f"  Malignant (1): {(stratified_df['label'] == 1).sum()}")
print(f"  Benign (0):    {(stratified_df['label'] == 0).sum()}")

print(f"\n👥 Patient Distribution:")
print(f"  Total patients: {stratified_df['study_id'].nunique()}")

print(f"\n🏥 BI-RADS Distribution:")
print(stratified_df['breast_birads'].value_counts())

# Show sample
print(f"\n📄 Sample rows:")
stratified_df.head()

## Step 6: DICOM → PNG Preprocessing

Convert DICOM files to preprocessed PNG using the mammogram preprocessing pipeline.

**This will:**
1. Load DICOM files
2. Apply ROI detection
3. Denoise (bilateral filter)
4. Enhance contrast (CLAHE)
5. Resize to 512×512
6. Normalize
7. Save as PNG

In [ ]:
# Import preprocessing pipeline
import sys
sys.path.insert(0, '/content/mammography-multiobjective-optimization')

from mammogram_preprocessor import MammogramPreprocessor
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# Initialize preprocessor
preprocessor = MammogramPreprocessor(target_size=512, clahe_clip_limit=2.0)

# Create output directory
output_dir = Path(PREPROCESSED_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Preprocessor initialized")
print(f"📁 Output directory: {output_dir}")

In [ ]:
# Preprocess all DICOM files
print("🔄 Converting DICOM → PNG...")
print(f"Total files to process: {len(stratified_df)}")
print("\nThis may take a while...\n")

processed_count = 0
skipped_count = 0
error_count = 0

# Add columns for PNG paths
stratified_df['png_path'] = None

for idx, row in tqdm(stratified_df.iterrows(), total=len(stratified_df)):
    # Build DICOM path
    dicom_path = Path(BASE_DIR) / row['file_path']
    
    # Build PNG path (preserve directory structure)
    relative_path = row['file_path'].replace('.dicom', '.png')
    png_path = output_dir / relative_path
    png_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Skip if already processed
    if png_path.exists():
        stratified_df.at[idx, 'png_path'] = str(png_path.relative_to(output_dir))
        skipped_count += 1
        continue
    
    try:
        # Preprocess DICOM
        preprocessed = preprocessor.process(str(dicom_path), verbose=False)
        
        # Convert to 0-255 range for PNG saving
        # Standardized image has mean~0, std~1
        # Clip to ±3 std and scale to 0-255
        img_clipped = np.clip(preprocessed, -3, 3)
        img_scaled = ((img_clipped + 3) / 6 * 255).astype(np.uint8)
        
        # Save as PNG
        Image.fromarray(img_scaled, mode='L').save(png_path)
        
        # Store relative path
        stratified_df.at[idx, 'png_path'] = str(png_path.relative_to(output_dir))
        processed_count += 1
        
    except Exception as e:
        print(f"\n⚠️  Error processing {dicom_path.name}: {e}")
        error_count += 1

print(f"\n" + "="*70)
print(f"✅ Preprocessing Complete!")
print(f"="*70)
print(f"  Processed: {processed_count}")
print(f"  Skipped (already exist): {skipped_count}")
print(f"  Errors: {error_count}")
print(f"  Total: {processed_count + skipped_count}")
print(f"\n📁 PNGs saved to: {output_dir}")
print(f"="*70)

In [ ]:
# Save updated metadata with PNG paths
metadata_with_png = output_dir / 'metadata_with_png.csv'
stratified_df.to_csv(metadata_with_png, index=False)

print(f"✅ Saved metadata with PNG paths: {metadata_with_png}")

## Step 7: Visualize Preprocessed Images

In [ ]:
import matplotlib.pyplot as plt

# Sample 10 images (5 malignant, 5 benign)
malignant_samples = stratified_df[stratified_df['label'] == 1].head(5)
benign_samples = stratified_df[stratified_df['label'] == 0].head(5)
samples = pd.concat([malignant_samples, benign_samples])

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Preprocessed Mammograms (512×512, Normalized)', fontsize=16, fontweight='bold')

for idx, (ax, (_, row)) in enumerate(zip(axes.flatten(), samples.iterrows())):
    png_path = output_dir / row['png_path']
    
    if png_path.exists():
        img = Image.open(png_path)
        ax.imshow(img, cmap='gray')
        
        label_name = "MALIGNANT" if row['label'] == 1 else "BENIGN"
        color = 'red' if row['label'] == 1 else 'green'
        
        title = f"[{idx+1}] {label_name}\n{row['breast_birads']}"
        ax.set_title(title, fontsize=10, color=color, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
    
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✅ All images are preprocessed and ready for training!")

## Step 8: Patient-Level Train/Val/Test Split

In [ ]:
import numpy as np

# Patient-level split
np.random.seed(42)

patients = stratified_df['study_id'].unique()
np.random.shuffle(patients)

# 70% train, 15% val, 15% test
n_train = int(len(patients) * 0.70)
n_val = int(len(patients) * 0.15)

train_patients = patients[:n_train]
val_patients = patients[n_train:n_train + n_val]
test_patients = patients[n_train + n_val:]

# Split dataframe
train_df = stratified_df[stratified_df['study_id'].isin(train_patients)].reset_index(drop=True)
val_df = stratified_df[stratified_df['study_id'].isin(val_patients)].reset_index(drop=True)
test_df = stratified_df[stratified_df['study_id'].isin(test_patients)].reset_index(drop=True)

print("="*70)
print("📊 PATIENT-LEVEL SPLIT")
print("="*70)
print(f"\nTrain: {len(train_patients)} patients, {len(train_df)} images")
print(f"  Malignant: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"  Benign:    {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).sum()/len(train_df)*100:.1f}%)")

print(f"\nVal: {len(val_patients)} patients, {len(val_df)} images")
print(f"  Malignant: {(val_df['label'] == 1).sum()} ({(val_df['label'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"  Benign:    {(val_df['label'] == 0).sum()} ({(val_df['label'] == 0).sum()/len(val_df)*100:.1f}%)")

print(f"\nTest: {len(test_patients)} patients, {len(test_df)} images")
print(f"  Malignant: {(test_df['label'] == 1).sum()} ({(test_df['label'] == 1).sum()/len(test_df)*100:.1f}%)")
print(f"  Benign:    {(test_df['label'] == 0).sum()} ({(test_df['label'] == 0).sum()/len(test_df)*100:.1f}%)")
print("="*70)

# Save splits
train_df.to_csv(output_dir / 'train.csv', index=False)
val_df.to_csv(output_dir / 'val.csv', index=False)
test_df.to_csv(output_dir / 'test.csv', index=False)

print(f"\n✅ Splits saved to: {output_dir}/")

## Step 9: Prepare for Optimization

Now we need to create a metadata format compatible with the optimization code.

In [ ]:
# Create optimization-compatible metadata
def prepare_optimization_metadata(df, image_base_dir):
    """
    Convert stratified metadata to optimization format.
    
    Required columns:
    - patient_id (or study_id)
    - breast_id
    - image_path
    - view
    - label
    - laterality (optional)
    """
    opt_df = df.copy()
    
    # Rename columns to match optimization code
    opt_df = opt_df.rename(columns={
        'study_id': 'patient_id',
        'view_position': 'view',
        'png_path': 'image_path'
    })
    
    # Create breast_id (patient + laterality)
    if 'laterality' in opt_df.columns:
        opt_df['breast_id'] = opt_df['patient_id'] + '_' + opt_df['laterality']
    else:
        opt_df['breast_id'] = opt_df['patient_id'] + '_Unknown'
    
    # Ensure required columns exist
    required = ['patient_id', 'breast_id', 'image_path', 'view', 'label']
    for col in required:
        if col not in opt_df.columns:
            raise ValueError(f"Missing required column: {col}")
    
    return opt_df

# Prepare metadata
train_opt = prepare_optimization_metadata(train_df, PREPROCESSED_DIR)
val_opt = prepare_optimization_metadata(val_df, PREPROCESSED_DIR)

print("✅ Optimization metadata prepared")
print(f"\nTrain: {len(train_opt)} images")
print(f"Val:   {len(val_opt)} images")
print(f"\nColumns: {list(train_opt.columns)}")
print(f"\nSample:")
train_opt.head()

## Step 10: Run NSGA-III Optimization

**Note:** Update the optimization code to use PNG images instead of DICOM.

In [ ]:
# Apply custom configuration to optimization
import os
import sys

# Update config.py with selected values
sys.path.insert(0, '/content/mammography-multiobjective-optimization')

import config

# Apply configuration from the preset
config.BATCH_SIZE = config['batch_size']
config.MAX_EPOCHS = config['max_epochs']
config.EARLY_STOPPING_PATIENCE = config['early_stopping_patience']
config.NSGA3_CONFIG['pop_size'] = config['pop_size']
config.NSGA3_CONFIG['n_generations'] = config['n_generations']

print("=" * 70)
print("🎛️ APPLYING CUSTOM CONFIGURATION")
print("=" * 70)
print(f"\nUpdated config.py with:")
print(f"  BATCH_SIZE = {config.BATCH_SIZE}")
print(f"  MAX_EPOCHS = {config.MAX_EPOCHS}")
print(f"  EARLY_STOPPING_PATIENCE = {config.EARLY_STOPPING_PATIENCE}")
print(f"  NSGA3_CONFIG['pop_size'] = {config.NSGA3_CONFIG['pop_size']}")
print(f"  NSGA3_CONFIG['n_generations'] = {config.NSGA3_CONFIG['n_generations']}")
print("=" * 70)

# Setup optimization directories
os.makedirs(f"{OPTIMIZATION_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{OPTIMIZATION_DIR}/results", exist_ok=True)

print("\n" + "=" * 70)
print("🚀 READY FOR OPTIMIZATION")
print("=" * 70)
print(f"\nDataset:")
print(f"  Training images:   {len(train_opt)}")
print(f"  Validation images: {len(val_opt)}")
print(f"  Image directory:   {PREPROCESSED_DIR}")
print(f"\nOptimization Settings:")
print(f"  Population Size:   {config.NSGA3_CONFIG['pop_size']}")
print(f"  Generations:       {config.NSGA3_CONFIG['n_generations']}")
print(f"  Max Epochs:        {config.MAX_EPOCHS}")
print(f"  Batch Size:        {config.BATCH_SIZE}")
print(f"\nOutput:")
print(f"  Checkpoints: {OPTIMIZATION_DIR}/checkpoints")
print(f"  Results:     {OPTIMIZATION_DIR}/results")
print("=" * 70)

# Import optimization runner
from optimization.nsga3_runner import NSGA3Runner

# Create runner
runner = NSGA3Runner(
    train_metadata=train_opt,
    val_metadata=val_opt,
    image_dir=PREPROCESSED_DIR,
    output_dir=f"{OPTIMIZATION_DIR}/results",
    checkpoint_dir=f"{OPTIMIZATION_DIR}/checkpoints",
    save_frequency=1
)

print("\n✅ NSGA3Runner initialized with custom configuration")
print("\nRun the next cell to start optimization!")

In [ ]:
# Start optimization
print("🚀 Starting NSGA-III optimization...\n")

result = runner.run()

print("\n" + "="*70)
print("✅ OPTIMIZATION COMPLETE!")
print("="*70)
print(f"\nPareto front size: {len(result.F)}")
print(f"Results saved to: {runner.output_dir}")
print("="*70)

## Step 11: Analyze Results

Load and visualize the Pareto front.

In [ ]:
import glob

# Find latest results
results_files = sorted(glob.glob(f"{OPTIMIZATION_DIR}/results/pareto_solutions_*.csv"))

if results_files:
    latest = results_files[-1]
    print(f"📊 Loading: {latest}")
    
    results_df = pd.read_csv(latest)
    print(f"\n✅ Loaded {len(results_df)} Pareto solutions")
    
    print("\n📈 Summary Statistics:")
    print(results_df[['pr_auc', 'auroc', 'brier', 'robustness_degradation']].describe())
    
    results_df.head()
else:
    print("❌ No results found. Run optimization first.")

In [ ]:
# Visualize Pareto front
if results_files:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Pareto Front - Objective Trade-offs', fontsize=16, fontweight='bold')
    
    pairs = [
        ('pr_auc', 'auroc'),
        ('pr_auc', 'brier'),
        ('pr_auc', 'robustness_degradation'),
        ('auroc', 'brier'),
        ('auroc', 'robustness_degradation'),
        ('brier', 'robustness_degradation')
    ]
    
    for ax, (x, y) in zip(axes.flatten(), pairs):
        ax.scatter(results_df[x], results_df[y], alpha=0.6, s=50, c='blue')
        ax.set_xlabel(x.replace('_', ' ').title(), fontsize=12)
        ax.set_ylabel(y.replace('_', ' ').title(), fontsize=12)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Summary

✅ **Complete workflow:**
1. Loaded stratified dataset from `stratified_selection.csv`
2. Converted DICOM → PNG with preprocessing pipeline
3. Patient-level train/val/test split
4. Ran NSGA-III optimization
5. Analyzed Pareto front

📁 **All files saved to Google Drive:**
- Preprocessed PNGs: `/content/drive/MyDrive/vindr-mammo/preprocessed_png_512/`
- Optimization results: `/content/drive/MyDrive/vindr_optimization/`

🚀 **Next steps:**
- Evaluate best solutions on test set
- Transfer to INbreast dataset
- Select final model based on priorities